# Fine-tuning Qwen2.5-1.5B với QLoRA cho Suy luận Pháp lý Tiếng Việt

Nghiên cứu này fine-tune mô hình ngôn ngữ nhỏ Qwen2.5-1.5B-Instruct cho ba dạng bài toán suy luận pháp lý tiếng Việt trong VLSP 2025 LegalSLM.
Chạy lần lượt các cell. Yêu cầu runtime **T4 GPU** trên Google Colab.


## 1. Kiểm tra môi trường


In [1]:
!nvidia-smi
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Không tìm thấy GPU")


Tue Apr 28 01:16:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers peft trl accelerate datasets rouge-score
!pip install -q --upgrade bitsandbytes
print("Cài đặt xong. Restart Runtime trước khi chạy tiếp (Runtime → Restart session).")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.0 MB/s eta 0:00:00
Cài đặt xong. Restart Runtime trước khi chạy tiếp (Runtime → Restart session).


## 2. Tải dữ liệu


In [3]:
from datasets import load_dataset
import random
from sklearn.model_selection import train_test_split

random.seed(42)

ds_mcq = load_dataset("VLSP2025-LegalSML/Public-Test", "multichoice_questions")
ds_nli = load_dataset("VLSP2025-LegalSML/Public-Test", "nli_questions")
ds_syl = load_dataset("VLSP2025-LegalSML/Public-Test", "syllogism_questions")

def get_split(ds):
    for split in ["train", "test", "validation"]:
        if split in ds:
            return list(ds[split])
    return list(ds[list(ds.keys())[0]])

mcq_data = get_split(ds_mcq)
nli_data  = get_split(ds_nli)
syl_data  = get_split(ds_syl)

print(f"MCQ: {len(mcq_data)} mẫu")
print(f"NLI: {len(nli_data)} mẫu")
print(f"Syllogism: {len(syl_data)} mẫu")
print(f"Tổng: {len(mcq_data) + len(nli_data) + len(syl_data)} mẫu")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

multichoice_questions/train-00000-of-000(…):   0%|          | 0.00/52.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/146 [00:00<?, ? examples/s]

nli_questions/train-00000-of-00001.parqu(…):   0%|          | 0.00/78.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/150 [00:00<?, ? examples/s]

syllogism_questions/train-00000-of-00001(…):   0%|          | 0.00/139k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/144 [00:00<?, ? examples/s]

MCQ: 146 mẫu
NLI: 150 mẫu
Syllogism: 144 mẫu
Tổng: 440 mẫu


## 3. Định dạng dữ liệu


In [4]:
SYSTEM_PROMPT = "Bạn là một chuyên gia pháp lý Việt Nam. Nhiệm vụ của bạn là phân tích văn bản pháp luật và trả lời các câu hỏi pháp lý một cách chính xác, súc tích."

def format_nli(sample):
    choices_text = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])])
    user_msg = (
        f"Văn bản pháp lý:\n{sample['legal_document']}\n\n"
        f"Câu hỏi cụ thể: {sample['specific_question']}\n\n"
        f"{sample['question']}\n{choices_text}\n\n"
        "Hãy chọn đáp án đúng (chỉ trả lời bằng chữ cái A hoặc B)."
    )
    return {"system": SYSTEM_PROMPT, "user": user_msg,
            "assistant": chr(65 + sample["answer"]), "task": "nli"}

def format_mcq(sample):
    choices_text = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])])
    user_msg = (
        f"Câu hỏi pháp luật: {sample['question']}\n\n"
        f"{choices_text}\n\n"
        "Hãy chọn đáp án đúng (chỉ trả lời bằng chữ cái A, B, C hoặc D)."
    )
    return {"system": SYSTEM_PROMPT, "user": user_msg,
            "assistant": chr(65 + sample["answer"]), "task": "mcq"}

def format_syllogism(sample):
    user_msg = (
        f"Tình huống pháp lý:\n{sample['question']}\n\n"
        "Hãy phân tích tình huống trên theo các bước:\n"
        "1. Xác định điều khoản pháp luật áp dụng (Tiền đề lớn)\n"
        "2. Áp dụng vào tình huống cụ thể (Tiền đề nhỏ)\n"
        "3. Đưa ra kết luận"
    )
    return {"system": SYSTEM_PROMPT, "user": user_msg,
            "assistant": sample["answer"], "task": "syllogism"}

nli_formatted = [format_nli(s) for s in nli_data]
mcq_formatted = [format_mcq(s) for s in mcq_data]
syl_formatted = [format_syllogism(s) for s in syl_data]

print(f"NLI: {len(nli_formatted)} | MCQ: {len(mcq_formatted)} | Syllogism: {len(syl_formatted)}")


NLI: 150 | MCQ: 146 | Syllogism: 144


In [5]:
nli_train, nli_test = train_test_split(nli_formatted, test_size=0.2, random_state=42)
mcq_train, mcq_test = train_test_split(mcq_formatted, test_size=0.2, random_state=42)
syl_train, syl_test = train_test_split(syl_formatted, test_size=0.2, random_state=42)

train_data = nli_train + mcq_train + syl_train
random.shuffle(train_data)

print(f"Train: NLI={len(nli_train)}, MCQ={len(mcq_train)}, Syllogism={len(syl_train)}, Tổng={len(train_data)}")
print(f"Test:  NLI={len(nli_test)},  MCQ={len(mcq_test)},  Syllogism={len(syl_test)},  Tổng={len(nli_test)+len(mcq_test)+len(syl_test)}")


Train: NLI=120, MCQ=116, Syllogism=115, Tổng=351
Test:  NLI=30,  MCQ=30,  Syllogism=29,  Tổng=89


## 4. Tải mô hình Qwen2.5-1.5B


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

try:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config,
        device_map="auto", trust_remote_code=True
    )
    print("Tải thành công với 4-bit QLoRA")
except Exception as e:
    print(f"Lỗi 4-bit: {e} — chuyển sang float16")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16,
        device_map="auto", trust_remote_code=True
    )
    print("Tải thành công với float16")

print(f"Số tham số: {model.num_parameters():,}")
print(f"Device: {next(model.parameters()).device}")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Tải thành công với 4-bit QLoRA
Số tham số: 1,543,714,304
Device: cuda:0


## 5. Đánh giá baseline (zero-shot)


In [7]:
import re
from rouge_score import rouge_scorer

def generate_response(model, tokenizer, system, user, max_new_tokens=256):
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

def extract_choice(text):
    text = text.strip().upper()
    m = re.search(r"^([ABCD])", text)
    if m:
        return m.group(1)
    for ch in ["A", "B", "C", "D"]:
        if ch in text[:10]:
            return ch
    return text[0] if text else "A"

def evaluate_classification(model, tokenizer, test_data, tag, max_tokens=16):
    correct, results = 0, []
    for i, sample in enumerate(test_data):
        response = generate_response(model, tokenizer, sample["system"], sample["user"], max_tokens)
        pred = extract_choice(response)
        gold = sample["assistant"]
        ok   = pred == gold
        correct += ok
        results.append({"pred": pred, "gold": gold, "correct": ok, "response": response})
        if (i + 1) % 5 == 0:
            print(f"  [{tag}] {i+1}/{len(test_data)} acc={correct/(i+1):.2%}")
    acc = correct / len(test_data)
    print(f"  [{tag}] Accuracy: {acc:.4f} ({correct}/{len(test_data)})")
    return acc, results

def evaluate_syllogism(model, tokenizer, test_data):
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    scores, results = [], []
    for i, sample in enumerate(test_data):
        response = generate_response(model, tokenizer, sample["system"], sample["user"], 512)
        rl = scorer.score(sample["assistant"], response)["rougeL"].fmeasure
        scores.append(rl)
        results.append({"pred": response, "gold": sample["assistant"], "rouge_l": rl})
        print(f"  [Syllogism] {i+1}/{len(test_data)} ROUGE-L={rl:.4f}")
    avg = sum(scores) / len(scores)
    print(f"  Avg ROUGE-L: {avg:.4f}")
    return avg, results

print("--- Zero-shot NLI ---")
zs_nli_acc, zs_nli_results   = evaluate_classification(model, tokenizer, nli_test, "NLI")
print("--- Zero-shot MCQ ---")
zs_mcq_acc, zs_mcq_results   = evaluate_classification(model, tokenizer, mcq_test, "MCQ")
print("--- Zero-shot Syllogism ---")
zs_syl_rouge, zs_syl_results = evaluate_syllogism(model, tokenizer, syl_test)


The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--- Zero-shot NLI ---
  [NLI] 5/30 acc=20.00%
  [NLI] 10/30 acc=60.00%
  [NLI] 15/30 acc=60.00%
  [NLI] 20/30 acc=70.00%
  [NLI] 25/30 acc=72.00%
  [NLI] 30/30 acc=66.67%
  [NLI] Accuracy: 0.6667 (20/30)
--- Zero-shot MCQ ---
  [MCQ] 5/30 acc=80.00%
  [MCQ] 10/30 acc=70.00%
  [MCQ] 15/30 acc=73.33%
  [MCQ] 20/30 acc=70.00%
  [MCQ] 25/30 acc=72.00%
  [MCQ] 30/30 acc=73.33%
  [MCQ] Accuracy: 0.7333 (22/30)
--- Zero-shot Syllogism ---
  [Syllogism] 1/29 ROUGE-L=0.4347
  [Syllogism] 2/29 ROUGE-L=0.3675
  [Syllogism] 3/29 ROUGE-L=0.4810
  [Syllogism] 4/29 ROUGE-L=0.4118
  [Syllogism] 5/29 ROUGE-L=0.4078
  [Syllogism] 6/29 ROUGE-L=0.3414
  [Syllogism] 7/29 ROUGE-L=0.3558
  [Syllogism] 8/29 ROUGE-L=0.4024
  [Syllogism] 9/29 ROUGE-L=0.4771
  [Syllogism] 10/29 ROUGE-L=0.3812
  [Syllogism] 11/29 ROUGE-L=0.3920
  [Syllogism] 12/29 ROUGE-L=0.4045
  [Syllogism] 13/29 ROUGE-L=0.3929
  [Syllogism] 14/29 ROUGE-L=0.4212
  [Syllogism] 15/29 ROUGE-L=0.4232
  [Syllogism] 16/29 ROUGE-L=0.4624
  [Syllogism]

## 6. Cấu hình LoRA và chuẩn bị dữ liệu huấn luyện


In [8]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [9]:
def format_for_sft(sample):
    messages = [
        {"role": "system",    "content": sample["system"]},
        {"role": "user",      "content": sample["user"]},
        {"role": "assistant", "content": sample["assistant"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_texts   = [format_for_sft(s) for s in train_data]
train_dataset = Dataset.from_dict({"text": train_texts})

print(f"Số mẫu huấn luyện: {len(train_dataset)}")
print(train_texts[0][:400])


Số mẫu huấn luyện: 351
<|im_start|>system
Bạn là một chuyên gia pháp lý Việt Nam. Nhiệm vụ của bạn là phân tích văn bản pháp luật và trả lời các câu hỏi pháp lý một cách chính xác, súc tích.<|im_end|>
<|im_start|>user
Câu hỏi pháp luật: Theo quy định pháp luật hiện hành, người nộp thuế có nghĩa vụ gì liên quan đến việc ghi mã số thuế trên hóa đơn khi thực hiện giao dịch kinh doanh?

A. Người nộp thuế phải ghi mã số thuế


## 7. Huấn luyện QLoRA


In [10]:
import time, inspect
from trl import SFTTrainer, SFTConfig

valid_sft_config  = inspect.signature(SFTConfig.__init__).parameters
valid_sft_trainer = inspect.signature(SFTTrainer.__init__).parameters

base_args = dict(
    output_dir="./qwen_legal_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    fp16=False,
    bf16=False,
    logging_steps=5,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none",
    dataloader_num_workers=0,
)

for key, val in [("max_seq_length", 1024), ("max_length", 1024),
                 ("dataset_text_field", "text"), ("packing", False)]:
    if key in valid_sft_config:
        base_args[key] = val

sft_config = SFTConfig(**base_args)

trainer_kwargs = dict(model=model, train_dataset=train_dataset, args=sft_config)
if "processing_class" in valid_sft_trainer:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in valid_sft_trainer:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

start_time   = time.time()
train_result = trainer.train()
training_time = (time.time() - start_time) / 60

print(f"Thời gian: {training_time:.1f} phút")
print(f"Loss cuối: {train_result.training_loss:.4f}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/351 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/351 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
5,1.667972
10,1.253327
15,1.013064
20,0.916761
25,0.886520
30,0.844026
35,0.847117
40,0.853864
45,0.787668
50,0.765371


Thời gian: 18.0 phút
Loss cuối: 0.9349


## 8. Đánh giá sau fine-tuning


In [11]:
model.eval()

print("--- Fine-tuned NLI ---")
ft_nli_acc, ft_nli_results   = evaluate_classification(model, tokenizer, nli_test, "NLI-FT")
print("--- Fine-tuned MCQ ---")
ft_mcq_acc, ft_mcq_results   = evaluate_classification(model, tokenizer, mcq_test, "MCQ-FT")
print("--- Fine-tuned Syllogism ---")
ft_syl_rouge, ft_syl_results = evaluate_syllogism(model, tokenizer, syl_test)


--- Fine-tuned NLI ---
  [NLI-FT] 5/30 acc=100.00%
  [NLI-FT] 10/30 acc=100.00%
  [NLI-FT] 15/30 acc=100.00%
  [NLI-FT] 20/30 acc=100.00%
  [NLI-FT] 25/30 acc=100.00%
  [NLI-FT] 30/30 acc=93.33%
  [NLI-FT] Accuracy: 0.9333 (28/30)
--- Fine-tuned MCQ ---
  [MCQ-FT] 5/30 acc=100.00%
  [MCQ-FT] 10/30 acc=90.00%
  [MCQ-FT] 15/30 acc=86.67%
  [MCQ-FT] 20/30 acc=90.00%
  [MCQ-FT] 25/30 acc=88.00%
  [MCQ-FT] 30/30 acc=86.67%
  [MCQ-FT] Accuracy: 0.8667 (26/30)
--- Fine-tuned Syllogism ---
  [Syllogism] 1/29 ROUGE-L=0.5696
  [Syllogism] 2/29 ROUGE-L=0.2220
  [Syllogism] 3/29 ROUGE-L=0.6222
  [Syllogism] 4/29 ROUGE-L=0.4894
  [Syllogism] 5/29 ROUGE-L=0.4956
  [Syllogism] 6/29 ROUGE-L=0.4475
  [Syllogism] 7/29 ROUGE-L=0.3124
  [Syllogism] 8/29 ROUGE-L=0.4444
  [Syllogism] 9/29 ROUGE-L=0.6124
  [Syllogism] 10/29 ROUGE-L=0.3538
  [Syllogism] 11/29 ROUGE-L=0.5808
  [Syllogism] 12/29 ROUGE-L=0.5395
  [Syllogism] 13/29 ROUGE-L=0.4818
  [Syllogism] 14/29 ROUGE-L=0.4336
  [Syllogism] 15/29 ROUGE-L=0.19

## 9. Load lại model nếu runtime bị reset

Nếu Colab bị ngắt sau khi huấn luyện xong, load adapter đã lưu


In [12]:
import torch, os, glob
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME   = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "./qwen_legal_lora"

checkpoints  = sorted(glob.glob(f"{ADAPTER_PATH}/checkpoint-*"))
adapter_path = checkpoints[-1] if checkpoints else ADAPTER_PATH
print(f"Adapter: {adapter_path}")

try:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config,
        device_map="auto", trust_remote_code=True
    )
except Exception as e:
    print(f"Lỗi 4-bit: {e} — dùng float16")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16,
        device_map="auto", trust_remote_code=True
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()
print("Sẵn sàng")


Adapter: ./qwen_legal_lora/checkpoint-66


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Sẵn sàng


## 10. Tổng hợp kết quả


In [13]:
sep = "-" * 64
print(sep)
print(f"{'Phương pháp':<30} {'NLI Acc':>10} {'MCQ Acc':>10} {'Syl ROUGE-L':>12}")
print(sep)
print(f"{'Zero-shot':<30} {zs_nli_acc:>10.4f} {zs_mcq_acc:>10.4f} {zs_syl_rouge:>12.4f}")
print(f"{'QLoRA (đề xuất)':<30} {ft_nli_acc:>10.4f} {ft_mcq_acc:>10.4f} {ft_syl_rouge:>12.4f}")
print(sep)

nli_imp = (ft_nli_acc   - zs_nli_acc)   / max(zs_nli_acc,   1e-6) * 100
mcq_imp = (ft_mcq_acc   - zs_mcq_acc)   / max(zs_mcq_acc,   1e-6) * 100
syl_imp = (ft_syl_rouge - zs_syl_rouge) / max(zs_syl_rouge, 1e-6) * 100
print(f"{'Cải thiện (%)':<30} {nli_imp:>+10.1f}% {mcq_imp:>+9.1f}% {syl_imp:>+11.1f}%")
print(sep)

print(f"\nNLI:       zero-shot={zs_nli_acc:.4f}  →  fine-tuned={ft_nli_acc:.4f}  ({nli_imp:+.1f}%)")
print(f"MCQ:       zero-shot={zs_mcq_acc:.4f}  →  fine-tuned={ft_mcq_acc:.4f}  ({mcq_imp:+.1f}%)")
print(f"Syllogism: zero-shot={zs_syl_rouge:.4f}  →  fine-tuned={ft_syl_rouge:.4f}  ({syl_imp:+.1f}%)")
print(f"Thời gian huấn luyện: {training_time:.1f} phút | Loss: {train_result.training_loss:.4f}")


----------------------------------------------------------------
Phương pháp                       NLI Acc    MCQ Acc  Syl ROUGE-L
----------------------------------------------------------------
Zero-shot                          0.6667     0.7333       0.4230
QLoRA (đề xuất)                    0.9333     0.8667       0.4827
----------------------------------------------------------------
Cải thiện (%)                       +40.0%     +18.2%       +14.1%
----------------------------------------------------------------

NLI:       zero-shot=0.6667  →  fine-tuned=0.9333  (+40.0%)
MCQ:       zero-shot=0.7333  →  fine-tuned=0.8667  (+18.2%)
Syllogism: zero-shot=0.4230  →  fine-tuned=0.4827  (+14.1%)
Thời gian huấn luyện: 18.0 phút | Loss: 0.9349


In [14]:
import shutil
from google.colab import files

model.save_pretrained("./qwen_legal_lora_final")
tokenizer.save_pretrained("./qwen_legal_lora_final")

shutil.make_archive("qwen_legal_lora_final", "zip", "./qwen_legal_lora_final")
files.download("qwen_legal_lora_final.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>